### Fama French Model Decomposition

Paul Grajzl

Friday, November 13
___

In [16]:
import sys
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from src import data, factors, plotting

# cached data
prices = data.load_prices().dropna(how="all")
returns = data.load_returns().dropna(how="all")
book = data.load_fundamental("book_equity")
shares_all = pd.read_parquet(data.fund_dir / "shares_all.parquet")

# Fama-French exposures
z_size = factors.standardize(factors.size(prices, shares_all))
z_val = factors.standardize(factors.value(book, prices, shares_all))
z_mom = factors.standardize(factors.momentum(prices))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Fama-French Factors: Background

**Three-factor model (Fama and French, 1993):**

$$
r_{i,t} - r_{f,t} = \alpha_i + \beta_i \,(\text{Mkt} - r_f)_t + s_i \, \text{SMB}_t + h_i \, \text{HML}_t + \varepsilon_{i,t}
$$

- **Mkt − RF**: market return above the risk-free rate (the CAPM factor)
- **SMB** (Small Minus Big): small stocks minus big stocks, the size effect
- **HML** (High Minus Low): high book-to-market (value) minus low book-to-market (growth), the value effect

Motivation: CAPM couldn't explain why small and value stocks historically earned more than their market betas implied. SMB and HML absorbed most of these anomalies.

**Extensions:**
- **Carhart four-factor (1997)**: adds momentum (**UMD** / **MOM**), built on 12-1 month returns
- **Fama-French five-factor (2015)**: adds **RMW** (robust minus weak profitability) and **CMA** (conservative minus aggressive investment, i.e. low vs high asset growth)

**Construction (portfolio sorts):**
1. Each June, sort all US stocks (NYSE, AMEX, Nasdaq) by size (NYSE median) and book-to-market (30th / 70th percentiles)
2. Form 6 value-weighted portfolios (2 size × 3 value)
3. **SMB** = average of 3 small − average of 3 big; **HML** = average of 2 high-B/M − average of 2 low-B/M
4. Book equity from the prior fiscal year, lagged at least six months (their point-in-time safeguard)

Daily factor returns are published free on Kenneth French's Data Library, back to 1926.

## Classic Fama-French vs Our Barra-Style Version

**The core flip:** Fama-French **observes** factor returns (sorted portfolios) and **estimates** each stock's exposures by time-series regression. Barra-style **knows** each stock's exposures (its characteristics) and **estimates** factor returns by cross-sectional regression each day.

| | Classic Fama-French | Our Barra-style version |
|---|---|---|
| Factor returns | Returns of long-short sorted portfolios | Daily cross-sectional regression slopes |
| Stock exposures | Time-series betas on the factors | Z-scored characteristics (log size, log B/P, 12-1 momentum) |
| Exposure shape | Coarse buckets (median, terciles) | Continuous z-scores: every stock's exact position counts |
| Weighting | Value-weighted portfolios | Equal-weighted OLS (every stock counts the same) |
| Controls | Double sort controls size within value only | Joint regression: each factor is pure, net of all others |
| Rebalancing | Annually (June) | Daily |
| Point-in-time | Fixed 6-month accounting lag | SEC filing dates |
| Value measure | Book equity / December market cap | Log book equity / current market cap |
| Market factor | Value-weighted market minus T-bills | Cross-sectional intercept: return of an average, style-neutral stock |
| Universe | All US stocks, NYSE breakpoints | Today's S&P 500 (survivorship bias) |

**Consequences to keep in mind:**
- Our factor returns are **pure**: e.g. our value return is net of size and momentum, while HML is only size-neutral
- Our results lean toward **large caps** (S&P only) and weight **every stock equally**, so they won't match the published factors exactly; correlation with them is the sanity check
- Profitability (RMW) and investment (CMA) need EDGAR income statement and asset-growth data before the five-factor version is complete

In [7]:
ff_exp = {"Size": z_size, "Value": z_val, "Momentum": z_mom}
res_ff = factors.cs_regression_multi(returns, ff_exp)            # intercept = market

ff = pd.DataFrame({
    "Market": res_ff["intercept"],
    "SMB": -res_ff["Size"],
    "HML": res_ff["Value"],
    "UMD": res_ff["Momentum"],
})

plotting.plot_factor_returns(ff["Market"], name="Market")

In [8]:
plotting.plot_factor_compare(
    {c: ff[c] for c in ["SMB", "HML", "UMD"]},
    title="Barra-style Fama-French factors (S&P 500)",
)

In [9]:
plotting.plot_table(
    factors.factor_stats(ff), title="Barra-style Fama-French + momentum: factor returns",
    index_name="Factor", fmt={"Days": "{:,.0f}"},
)

In [12]:
ff_off = data.get_ff_factors()
ff_off["Mkt"] = ff_off["Mkt-RF"] + ff_off["RF"]          # total market return, to match ours
ff_off = ff_off.add_prefix("FF_")

pairs = {"Market": "FF_Mkt", "SMB": "FF_SMB", "HML": "FF_HML", "UMD": "FF_Mom"}
both = ff.join(ff_off, how="inner")

corr = pd.Series({ours: both[ours].corr(both[off]) for ours, off in pairs.items()},
                 name="Correlation with official")
plotting.plot_table(corr, title="Our Barra-style factors vs official Fama-French",
                    subtitle=f"{both.index.min():%b %Y} to {both.index.max():%b %Y}, daily",
                    index_name="Factor")

In [14]:
plotting.plot_factor_compare(
    {"SMB (ours)": both["SMB"], "SMB (official)": both["FF_SMB"]},
    title="SMB: ours vs official",
)

### Barra-Style Fama-French: Results and Validation

**Model:** daily cross-sectional regression of S&P 500 returns on z-scored size, value and momentum exposures, no sectors. The intercept is the market factor.

$$
r_{i,t} = f^{\text{mkt}}_t + f^{\text{size}}_t \, z^{\text{size}}_{i} + f^{\text{value}}_t \, z^{\text{value}}_{i} + f^{\text{mom}}_t \, z^{\text{mom}}_{i} + \varepsilon_{i,t}
$$

Mapped to Fama-French conventions: **SMB = −size** (small minus big), **HML = value**, **UMD = momentum**.

**Factor returns (~4 years, 1,000 days):**

| Factor | Ann. return | Ann. vol | Sharpe | t-stat |
|---|---|---|---|---|
| Market | 17.9% | 14.7% | 1.22 | 2.43 |
| SMB | 4.3% | 3.1% | 1.38 | 2.75 |
| HML | 2.1% | 3.3% | 0.62 | 1.24 |
| UMD | 7.5% | 8.5% | 0.88 | 1.75 |

**Daily correlation with official Fama-French factors:** Market 0.91, SMB 0.61, HML 0.71, UMD 0.87. Built independently from raw SEC filings and prices, the factors clearly capture the same dimensions.

**Why they differ:**
- **Universe**: S&P 500 only (mid vs mega cap) vs all US stocks (micro vs mega cap). This hits SMB hardest.
- **Purity**: ours are net of each other via the joint regression; HML is only size-neutral.
- **Survivorship**: today's S&P includes small companies that grew into it, biasing our SMB positive even though official SMB has been weak.
- **Correlation ≠ average return**: factors can move together day to day while drifting apart in level.

**Scale: per σ vs portfolio returns.** Our factor returns are regression slopes: the return **per 1σ of exposure**, spread across ~500 large caps with equal weights, hence highly diversified and low-vol. Official factors are returns of **long-short portfolios** of extreme stocks (top vs bottom 30%), across all US stocks including volatile small caps, so they're more volatile. The two live on different scales: **compare correlations and shapes, not magnitudes**.

___

### Replacing S&P with Russell 3000
___

In [21]:
universe = data.get_russell3000()
len(universe), universe["sector"].nunique()

(2565, 11)

In [22]:
prices, volume, failed = data.get_prices_volume(universe["ticker"].tolist())
returns = data.compute_returns(prices)
spy_px, spy_ret = data.get_market("SPY")          # needed for beta later

prices.shape, len(failed), failed[:20]

200/2565
400/2565
600/2565
800/2565
1000/2565
1200/2565
1400/2565
1600/2565
1800/2565
2000/2565
2200/2565
2400/2565
2565/2565


((1255, 2565), 0, [])

### Return Cleaning and the Estimation Universe

### The problem

Expanding from the S&P 500 to the Russell 3000 brings in thousands of small caps, and with them extreme daily returns. The largest absolute daily returns in the raw data were:

- **Data errors**: DEC showed an infinite return (prior price recorded as zero), followed by a +1,900% day. Some other huge one-day moves are likely single bad prints.
- **Real but extreme moves**: small biotechs on trial news, small caps pivoting to crypto treasuries, short squeezes. Moves of +400% to +700% in a day are genuine for these names.

Both are dangerous in a regression. OLS minimizes **squared** residuals, so one +500% day carries the weight of thousands of normal observations and can drag that day's factor returns far more than every other stock combined.

### Estimation universe vs coverage universe

Risk models separate two sets of stocks:

- **Estimation universe**: the clean, reasonably liquid stocks used to **estimate factor returns** each day
- **Coverage universe**: every stock the model is **applied** to (exposures, attribution, risk)

A stock can be excluded from estimation but still be covered: its exposures still exist, and its return can still be decomposed using factor returns estimated from everyone else.

### Cleaning steps

**1. Remove impossible prices.** A price of zero or below is never valid. These are set to missing before computing returns, which removes infinite returns.

**2. Price filter (≥ $5, prior close).** Penny stocks have the noisiest prices and most bad prints, and many extreme movers traded below $5 before their spikes. The filter uses the **prior day's** price, so it never uses information from the day being estimated.

**3. Cap daily returns at ±50% for estimation.** Genuine big movers stay in the regression, but no single stock can dominate a day. Moves beyond ±50% are almost always stock-specific news, i.e. idio, so capping them loses essentially no information about factor returns.

**4. Keep raw returns for attribution.** When decomposing a specific stock, use its **actual** return: factor contributions come from the factor returns, and everything else, including any extreme move, lands in idio.

### Summary

| Step | Applies to | Purpose |
|---|---|---|
| Price ≤ 0 → missing | All data | Remove impossible values |
| Prior price ≥ $5 | Estimation only | Exclude penny stocks |
| Returns capped at ±50% | Estimation only | Stop single stocks dominating |
| Raw returns kept | Attribution | Decompose actual performance |

**Expected size**: roughly 2,000 to 2,300 stocks per day in the estimation universe.